# Notebook 08: Evaluation, Reliability, and Probability Calibration
**GlobalFlood AI: Global Multimodal Flood Prediction System (48-Hour Forecast)**

### Objective
Perform scientific validation on the held-out test events:
1. **Model Evaluation:** ROC-AUC, PR-AUC, F1-score, Precision, Recall, Brier score, and Confusion Matrix
2. **Platt Scaling Calibration:** Fit univariate logistic calibration on validation logits to ensure predicted probabilities match empirical flood frequencies
3. **Application Risk Tiers:** Map calibrated probabilities to Low (<25%), Moderate (25-55%), High (55-80%), and Very High (>=80%)

### Inputs & Outputs
- **Inputs:** Trained model checkpoint and test set
- **Outputs:** Saved calibrator `models/checkpoints/calibrator.joblib` and scientific evaluation report
- **Dependencies:** `scikit-learn`, `ml.src.evaluate`, `ml.src.calibrate`


In [ ]:
import json
import numpy as np
import torch
from torch.utils.data import DataLoader
from ml.src.model import MultimodalFloodNet
from ml.src.dataset import MultimodalFloodDataset
from ml.src.evaluate import evaluate_model
from ml.src.calibrate import ProbabilityCalibrator
from ml.src.train import get_device
from app.core.config import settings

device = get_device()
weights_path = settings.MODEL_WEIGHTS_FILE

# Load model
ckpt = torch.load(weights_path, map_location=device)
model = MultimodalFloodNet().to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Load validation & test samples
with open(settings.PROCESSED_DATA_DIR / "val_samples.json") as f:
    val_samples = json.load(f)
with open(settings.PROCESSED_DATA_DIR / "test_samples.json") as f:
    test_samples = json.load(f)

# 1. Fit Platt Scaling Calibrator on Validation Predictions
val_ds = MultimodalFloodDataset(val_samples)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
val_probs, val_labels = [], []

with torch.no_grad():
    for b in val_loader:
        p = torch.sigmoid(model(
            b["sar_patch"].to(device), b["sar_mask"].to(device),
            b["hist_weather"].to(device), b["hist_weather_mask"].to(device),
            b["fc_weather"].to(device), b["fc_weather_mask"].to(device),
            b["geo_features"].to(device), b["geo_mask"].to(device)
        )).cpu().numpy()
        val_probs.extend(p.tolist())
        val_labels.extend(b["label"].numpy().tolist())

calibrator = ProbabilityCalibrator(method="platt")
calibrator.fit(np.array(val_probs), np.array(val_labels))
calibrator.save(settings.CALIBRATION_FILE)
print(f"Fitted and saved Platt calibrator to {settings.CALIBRATION_FILE}")


### 2. Scientific Evaluation on Held-Out Test Events
Strict evaluation on test events (Sylhet and Yorkshire) never seen during training or calibration.


In [ ]:
eval_results = evaluate_model(model, test_samples, device, is_smoke_test=False)
print("Held-Out Test Set Evaluation Report:")
print(json.dumps(eval_results, indent=2))

# Map probabilities to risk categories
def categorize_risk(prob: float) -> str:
    if prob < 0.25:
        return "Low (<25%)"
    elif prob < 0.55:
        return "Moderate (25-55%)"
    elif prob < 0.80:
        return "High (55-80%)"
    else:
        return "Very High (>=80%)"

print("\nSample Calibrated Inference:")
for sample_prob in [0.08, 0.35, 0.68, 0.91]:
    cal = calibrator.calibrate(sample_prob)
    print(f"Raw Prob: {sample_prob:.2f} -> Calibrated Prob: {cal:.2f} -> Category: {categorize_risk(cal)}")

print("\n[SUCCESS] Scientific evaluation and probability calibration completed successfully.")
